# Bank Marketing Campaign Classification with Logistic Regression

## 1. Environment Setup

In [41]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from datetime import datetime
import json
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Set seaborn style
sns.set_theme(style='whitegrid')

## 2. Utility Functions

In [42]:
def plot_confusion_matrix(cm, title='Confusion Matrix', save_path=None):
    """Generate and save confusion matrix plot"""
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Not Subscribed', 'Subscribed'],
                yticklabels=['Not Subscribed', 'Subscribed'])
    plt.title(title)
    plt.xlabel('Predicted Label')
    plt.ylabel('True Label')
    if save_path:
        plt.savefig(save_path)
        plt.close()
    else:
        plt.show()

def save_metrics(metrics_dict, filename):
    """Save evaluation metrics to JSON file"""
    with open(filename, 'w') as f:
        json.dump(metrics_dict, f, indent=4)

## 3. Data Preprocessing

In [43]:
def preprocess_data(df, target_column='y', 
                    dummy_cols=['job','marital','education','default','housing', 'loan','contact','month','poutcome'],
                    columns_alignment=None):
    """
    Preprocess dataset by:
    1. Converting categorical variables to dummies
    2. Converting target variable to binary
    3. Aligning columns if provided
    """
    df_proc = pd.get_dummies(df, columns=dummy_cols)
    df_proc[target_column] = df_proc[target_column].map({'yes': 1, 'no': 0})
    
    X = df_proc.drop(target_column, axis=1)
    y = df_proc[target_column]
    
    if columns_alignment is not None:
        X = X.reindex(columns=columns_alignment, fill_value=0)
    return X, y

## 4. Data Loading & Preparation

In [44]:
# Load training data
train_path = '../../../dataset/archive/bank.csv'
df_train = pd.read_csv(train_path, sep=';')

# Preprocess training data
X_train_full, y_train_full = preprocess_data(df_train)
training_columns = X_train_full.columns

# Split into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, 
    test_size=0.2, 
    random_state=42
)

# Feature scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

## 5. Model Training

In [45]:
# Initialize and train model
lr_model = LogisticRegression(random_state=42, max_iter=1000)
lr_model.fit(X_train_scaled, y_train)

LogisticRegression(max_iter=1000, random_state=42)

## 6. Model Evaluation

In [46]:
# Training set evaluation
train_preds = lr_model.predict(X_train_scaled)
print("Training Metrics:")
print(f"Accuracy: {accuracy_score(y_train, train_preds):.2%}")
print("\nClassification Report:\n", classification_report(y_train, train_preds))

Training Metrics:
Accuracy: 90.71%

Classification Report:
               precision    recall  f1-score   support

           0       0.92      0.98      0.95      3193
           1       0.69      0.38      0.49       423

    accuracy                           0.91      3616
   macro avg       0.80      0.68      0.72      3616
weighted avg       0.89      0.91      0.89      3616



In [47]:
# Validation set evaluation
val_preds = lr_model.predict(X_val_scaled)
print("Validation Metrics:")
print(f"Accuracy: {accuracy_score(y_val, val_preds):.2%}")
print("\nClassification Report:\n", classification_report(y_val, val_preds))

Validation Metrics:
Accuracy: 90.17%

Classification Report:
               precision    recall  f1-score   support

           0       0.92      0.98      0.95       807
           1       0.60      0.28      0.38        98

    accuracy                           0.90       905
   macro avg       0.76      0.63      0.66       905
weighted avg       0.88      0.90      0.88       905



## 7. Full Dataset Evaluation

In [48]:
# Load and process full dataset
full_path = '../../../dataset/archive/bank-full.csv'
df_full = pd.read_csv(full_path, sep=';')
X_full, y_full = preprocess_data(df_full, columns_alignment=training_columns)
X_full_scaled = scaler.transform(X_full)

# Full dataset evaluation
full_preds = lr_model.predict(X_full_scaled)
print("Full Dataset Metrics:")
print(f"Accuracy: {accuracy_score(y_full, full_preds):.2%}")
print("\nClassification Report:\n", classification_report(y_full, full_preds))

Full Dataset Metrics:
Accuracy: 90.18%

Classification Report:
               precision    recall  f1-score   support

           0       0.92      0.98      0.95     39922
           1       0.65      0.35      0.45      5289

    accuracy                           0.90     45211
   macro avg       0.78      0.66      0.70     45211
weighted avg       0.89      0.90      0.89     45211



## 8. Results Saving

In [49]:
# Generate timestamp
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

# Save metrics
metrics = {
    'training': {
        'accuracy': accuracy_score(y_train, train_preds),
        'classification_report': classification_report(y_train, train_preds, output_dict=True)
    },
    'validation': {
        'accuracy': accuracy_score(y_val, val_preds),
        'classification_report': classification_report(y_val, val_preds, output_dict=True)
    },
    'full_dataset': {
        'accuracy': accuracy_score(y_full, full_preds),
        'classification_report': classification_report(y_full, full_preds, output_dict=True)
    }
}
save_metrics(metrics, f'../metrics/logistic_regression_metrics_{timestamp}.json')

# Save confusion matrices
plot_confusion_matrix(confusion_matrix(y_train, train_preds), 
                     title='Training Set Confusion Matrix',
                     save_path=f'../plots/confusion_matrix_train_{timestamp}.png')

plot_confusion_matrix(confusion_matrix(y_val, val_preds),
                     title='Validation Set Confusion Matrix',
                     save_path=f'../plots/confusion_matrix_val_{timestamp}.png')

plot_confusion_matrix(confusion_matrix(y_full, full_preds),
                     title='Full Dataset Confusion Matrix',
                     save_path=f'../plots/confusion_matrix_full_{timestamp}.png')